In [1]:
import pandas as pd
import numpy as np
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

def NowCast(valores, PM):
    """
    Calcula el NowCast según NOM-172-SEMARNAT-2023.
    PM: 0 para PM10, 1 para PM2.5.
    """
    ultimas_3 = valores[-3:] if len(valores) >= 3 else valores
    if sum(x is not None and not pd.isna(x) for x in ultimas_3) < 2:
        return None

    valores_rev = valores[::-1]
    datos = []
    i = 0
    for v in valores_rev:
        if v is not None and not pd.isna(v):
            datos.append((float(v), i))
        i += 1

    if len(datos) < 2:
        return None

    solo_val = [v for v, _ in datos]
    rango = max(solo_val) - min(solo_val)
    w_raw = round(1 - (rango / max(solo_val)), 2) if max(solo_val) > 0 else 0.5
    W = w_raw if w_raw >= 0.5 else 0.5

    num = 0.0
    den = 0.0
    for v, hora_consec in datos:
        peso = (W ** hora_consec)
        num += v * peso
        den += peso

    if den == 0:
        return None

    promedio = round(num / den, 0)

    # Factores de ajuste Anexo A (NOM-172)
    if PM == 0:  # PM10
        promedio = round(promedio * 0.714, 0)
    else:       # PM2.5
        promedio = round(promedio * 0.694, 0)

    return int(promedio)


# Funciones de Categorización (NOM-172-SEMARNAT-2023)
def cat_pm10(val):
    if pd.isna(val) or val is None: return np.nan
    if val <= 50: return 'Buena'
    elif val <= 100: return 'Aceptable'
    elif val <= 155: return 'Mala'
    elif val <= 235: return 'Muy Mala'
    else: return 'Extremadamente Mala'

def cat_pm25(val):
    if pd.isna(val) or val is None: return np.nan
    if val <= 25: return 'Buena'
    elif val <= 45: return 'Aceptable'
    elif val <= 79: return 'Mala'
    elif val <= 147: return 'Muy Mala'
    else: return 'Extremadamente Mala'

def cat_o3(o3_1h, o3_8h):
    c1, c8 = np.nan, np.nan
    if not pd.isna(o3_1h):
        if o3_1h <= 0.051: c1 = 1
        elif o3_1h <= 0.070: c1 = 2
        elif o3_1h <= 0.092: c1 = 3
        elif o3_1h <= 0.114: c1 = 4
        else: c1 = 5
    if not pd.isna(o3_8h):
        if o3_8h <= 0.051: c8 = 1
        elif o3_8h <= 0.070: c8 = 2
        elif o3_8h <= 0.092: c8 = 3
        elif o3_8h <= 0.0114: c8 = 4
        else: c8 = 5
    peor = max([c for c in [c1, c8] if not pd.isna(c)], default=np.nan)
    mapping = {1: 'Buena', 2: 'Aceptable', 3: 'Mala', 4: 'Muy Mala', 5: 'Extremadamente Mala'}
    return mapping.get(peor, np.nan)

def cat_co(val):
    if pd.isna(val) or val is None: return np.nan
    if val <= 5.0: return 'Buena'
    elif val <= 9.5: return 'Aceptable'
    elif val <= 11.5: return 'Mala'
    elif val <= 13.5: return 'Muy Mala'
    else: return 'Extremadamente Mala'

def cat_no2(val):
    if pd.isna(val) or val is None: return np.nan
    if val <= 0.107: return 'Buena'
    elif val <= 0.210: return 'Aceptable'
    elif val <= 0.230: return 'Mala'
    elif val <= 0.250: return 'Muy Mala'
    else: return 'Extremadamente Mala'

def cat_so2(val):
    if pd.isna(val) or val is None: return np.nan
    if val <= 0.035: return 'Buena'
    elif val <= 0.075: return 'Aceptable'
    elif val <= 0.185: return 'Mala'
    elif val <= 0.304: return 'Muy Mala'
    else: return 'Extremadamente Mala'


def procesar_archivo_estacion(ruta_archivo, archivo_salida='BDPIN_Resultados_NOM172_Estilizado.xlsx'):
    print(f"Cargando y procesando datos desde: {ruta_archivo}...")

    # 1. Cargar hoja (detectar automáticamente la primera pestaña o usar 'Data')
    xls = pd.ExcelFile(ruta_archivo)
    sheet_name = 'Data' if 'Data' in xls.sheet_names else xls.sheet_names[0]
    df = pd.read_excel(ruta_archivo, sheet_name=sheet_name)
    df = df.sort_values(by=['DATE', 'HOUR']).reset_index(drop=True)

    # 2. Promedios Móviles
    df['PM10_24H'] = df['PM10'].rolling(window=24, min_periods=18).mean().round(0) if 'PM10' in df.columns else np.nan
    df['PM2.5_24H'] = df['PM2.5'].rolling(window=24, min_periods=18).mean().round(1) if 'PM2.5' in df.columns else np.nan
    df['CO_8H'] = df['CO'].rolling(window=8, min_periods=6).mean().round(2) if 'CO' in df.columns else np.nan
    df['O3_8H'] = df['O3'].rolling(window=8, min_periods=6).mean().round(3) if 'O3' in df.columns else np.nan

    # 3. NowCast deslizable (12 horas)
    pm10_nc, pm25_nc = [], []
    for idx in range(len(df)):
        sub_10 = df['PM10'].iloc[max(0, idx-11):idx+1].tolist() if 'PM10' in df.columns else []
        sub_25 = df['PM2.5'].iloc[max(0, idx-11):idx+1].tolist() if 'PM2.5' in df.columns else []

        if len(sub_10) < 12: sub_10 = [None]*(12-len(sub_10)) + sub_10
        if len(sub_25) < 12: sub_25 = [None]*(12-len(sub_25)) + sub_25

        pm10_nc.append(NowCast(sub_10, PM=0))
        pm25_nc.append(NowCast(sub_25, PM=1))

    df['PM10_NOWCAST'] = pm10_nc
    df['PM2.5_NOWCAST'] = pm25_nc

    # 4. Categorización por contaminante
    df['IAS_PM10_CAT'] = df['PM10_NOWCAST'].apply(cat_pm10)
    df['IAS_PM2.5_CAT'] = df['PM2.5_NOWCAST'].apply(cat_pm25)
    df['IAS_CO_CAT'] = df['CO_8H'].apply(cat_co) if 'CO_8H' in df.columns else np.nan
    df['IAS_O3_CAT'] = [cat_o3(r['O3'], r['O3_8H']) for _, r in df.iterrows()] if ('O3' in df.columns and 'O3_8H' in df.columns) else np.nan
    df['IAS_NO2_CAT'] = df['NO2'].apply(cat_no2) if 'NO2' in df.columns else np.nan
    df['IAS_SO2_CAT'] = df['SO2'].apply(cat_so2) if 'SO2' in df.columns else np.nan

    # 5. Índice Global y Poluyente Predominante
    orden_severidad = {'Extremadamente Mala': 5, 'Muy Mala': 4, 'Mala': 3, 'Aceptable': 2, 'Buena': 1}
    orden_inv = {5: 'Extremadamente Mala', 4: 'Muy Mala', 3: 'Mala', 2: 'Aceptable', 1: 'Buena'}

    global_cat, global_pol = [], []

    for idx, r in df.iterrows():
        cats = {
            'PM10': r.get('IAS_PM10_CAT', np.nan),
            'PM2.5': r.get('IAS_PM2.5_CAT', np.nan),
            'CO': r.get('IAS_CO_CAT', np.nan),
            'O3': r.get('IAS_O3_CAT', np.nan),
            'NO2': r.get('IAS_NO2_CAT', np.nan),
            'SO2': r.get('IAS_SO2_CAT', np.nan)
        }

        validos = {pol: cat for pol, cat in cats.items() if pd.notna(cat)}

        if not validos:
            global_cat.append(np.nan)
            global_pol.append(np.nan)
        else:
            max_score = -1
            peor_pols = []
            for pol, cat in validos.items():
                score = orden_severidad[cat]
                if score > max_score:
                    max_score = score
                    peor_pols = [pol]
                elif score == max_score:
                    peor_pols.append(pol)

            global_cat.append(orden_inv[max_score])
            global_pol.append(", ".join(peor_pols))

    df['IAS_GLOBAL_CAT'] = global_cat
    df['IAS_GLOBAL_POL'] = global_pol

    # 6. Orden de columnas exacto
    columnas_ordenadas = [
        'STATION', 'DATE', 'HOUR', 'O3', 'NO', 'NO2', 'NOX', 'SO2', 'CO', 'PM10', 'PM2.5',
        'PM10_NOWCAST', 'PM10_24H', 'PM2.5_NOWCAST', 'PM2.5_24H', 'CO_8H', 'O3_8H',
        'IAS_PM10_CAT', 'IAS_PM2.5_CAT', 'IAS_CO_CAT', 'IAS_O3_CAT', 'IAS_NO2_CAT', 'IAS_SO2_CAT',
        'IAS_GLOBAL_CAT', 'IAS_GLOBAL_POL'
    ]

    df_final = df[columnas_ordenadas]

    # ==========================================
    # 7. EXPORTACIÓN Y FORMATO CON OPENPYXL
    # ==========================================
    with pd.ExcelWriter(archivo_salida, engine='openpyxl') as writer:
        df_final.to_excel(writer, sheet_name='Aire_y_Salud', index=False)

    wb = openpyxl.load_workbook(archivo_salida)
    ws = wb['Aire_y_Salud']

    # Estilos principales
    font_header = Font(name='Segoe UI', size=11, bold=True, color='FFFFFF')
    fill_header = PatternFill(start_color='1F4E78', end_color='1F4E78', fill_type='solid')
    font_data = Font(name='Segoe UI', size=10)

    thin_border = Border(
        left=Side(style='thin', color='D9D9D9'), right=Side(style='thin', color='D9D9D9'),
        top=Side(style='thin', color='D9D9D9'), bottom=Side(style='thin', color='D9D9D9')
    )

    category_colors = {
        'Buena': {'fill': 'C6EFCE', 'font': '006100'},
        'Aceptable': {'fill': 'FFEB9C', 'font': '9C6500'},
        'Mala': {'fill': 'FCE4D6', 'font': 'C65911'},
        'Muy Mala': {'fill': 'FFC7CE', 'font': '9C0006'},
        'Extremadamente Mala': {'fill': 'E1D5E7', 'font': '622383'}
    }

    ws.freeze_panes = 'B2'
    ws.row_dimensions[1].height = 30

    # Aplicar formato a los encabezados
    for col_num in range(1, ws.max_column + 1):
        cell = ws.cell(row=1, column=col_num)
        cell.font = font_header
        cell.fill = fill_header
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

    # Aplicar formato a las celdas de datos
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
        for cell in row:
            cell.font = font_data
            cell.border = thin_border
            encabezado = str(ws.cell(row=1, column=cell.column).value)
            valor = str(cell.value) if cell.value is not None else ''

            # Alineación y formato numérico por tipo de columna
            if encabezado in ['STATION', 'DATE', 'HOUR']:
                cell.alignment = Alignment(horizontal='center')
                if encabezado == 'DATE':
                    cell.number_format = 'yyyy-mm-dd'

            elif 'CAT' in encabezado or encabezado == 'IAS_GLOBAL_POL':
                cell.alignment = Alignment(horizontal='center')
                if valor in category_colors:
                    cell.fill = PatternFill(start_color=category_colors[valor]['fill'],
                                            end_color=category_colors[valor]['fill'], fill_type='solid')
                    cell.font = Font(name='Segoe UI', size=10, bold=True, color=category_colors[valor]['font'])

            elif 'PM10' in encabezado or 'PM2.5' in encabezado:
                cell.alignment = Alignment(horizontal='right')
                cell.number_format = '0.0' if '24H' in encabezado and '2.5' in encabezado else '0'

            elif any(gas in encabezado for gas in ['O3', 'NO', 'NO2', 'NOX', 'SO2', 'CO']):
                cell.alignment = Alignment(horizontal='right')
                cell.number_format = '0.00' if 'CO' in encabezado else '0.000'

    # Autoajuste de ancho de columnas
    for col in ws.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        letra = get_column_letter(col[0].column)
        ws.column_dimensions[letra].width = min(max(max_len + 3, 12), 26)

    wb.save(archivo_salida)
    print(f"\n¡Proceso finalizado con éxito! Archivo bonito generado: '{archivo_salida}'")


# Ejecutar proceso
if __name__ == "__main__":
    procesar_archivo_estacion('BDPIN_Marzo_2024.xlsx')

Cargando y procesando datos desde: BDPIN_Marzo_2024.xlsx...

¡Proceso finalizado con éxito! Archivo bonito generado: 'BDPIN_Resultados_NOM172_Estilizado.xlsx'
